In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "462_build_gold_vw_fact_project_bid_role_comparison.py"
RUN_ID = str(uuid.uuid4())

SOURCE_TABLE = f"{catalog}.gold.fact_project_bid_role_comparison"
TARGET_VIEW = f"{catalog}.gold.vw_fact_project_bid_role_comparison"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.vw_fact_project_bid_role_comparison
# ============================================================
try:
    spark.sql(f"DROP VIEW IF EXISTS {TARGET_VIEW}")

    spark.sql(f"""
    CREATE VIEW {TARGET_VIEW} AS
    SELECT
          project_bid_role_context_key                        AS `Project Bid Role Comparison Key`
        , project_key                                         AS `Project Key`
        , project_id                                          AS `Project ID`

        , project_classification_key                          AS `Project Classification Key`
        , county_key                                          AS `County Key`

        , project_name                                        AS `Project Name`
        , project_number                                      AS `Project Number`
        , state_project_number                                AS `State Project Number`
        , control_section_job_csj                             AS `Control Section Job CSJ`
        , controlling_project_id_ccsj                         AS `Controlling Project ID CCSJ`
        , project_type                                        AS `Project Type`
        , project_classification                              AS `Project Classification`
        , CAST(project_actual_let_date AS DATE)               AS `Project Actual Let Date`
        , project_estimated_let_date                          AS `Project Estimated Let Date`
        , project_limits_from                                 AS `Project Limits From`
        , project_limits_to                                   AS `Project Limits To`
        , county                                              AS `County`
        , location                                            AS `Location`
        , district_division                                   AS `District Division`
        , highway                                             AS `Highway`
        , short_description                                   AS `Short Description`
        , federal_project_number                              AS `Federal Project Number`

        , configured_focus_vendor_name                        AS `Configured Focus Vendor Name`
        , configured_focus_vendor_name_standardized           AS `Configured Focus Vendor Name Standardized`

        , focus_vendor_present_flag                           AS `Focus Vendor Present Flag`
        , focus_vendor_won_flag                               AS `Focus Vendor Won Flag`
        , focus_vendor_lost_flag                              AS `Focus Vendor Lost Flag`
        , focus_vendor_competed_flag                          AS `Focus Vendor Competed Flag`
        , focus_vendor_key                                    AS `Focus Vendor Key`
        , focus_vendor_name                                   AS `Focus Vendor Name`
        , focus_vendor_name_standardized                      AS `Focus Vendor Name Standardized`
        , focus_bid_total_amount                              AS `Focus Bid Total Amount`
        , focus_project_bid_rank                              AS `Focus Project Bid Rank`
        , focus_vendor_status                                 AS `Focus Vendor Status`
        , focus_vendor_rank_bucket                            AS `Focus Vendor Rank Bucket`

        , winner_vendor_key                                   AS `Winner Vendor Key`
        , winner_vendor_name                                  AS `Winner Vendor Name`
        , winner_vendor_name_standardized                     AS `Winner Vendor Name Standardized`
        , winner_bid_total_amount                             AS `Winner Bid Total Amount`
        , winner_project_bid_rank                             AS `Winner Project Bid Rank`

        , second_vendor_key                                   AS `Second Vendor Key`
        , second_vendor_name                                  AS `Second Vendor Name`
        , second_vendor_name_standardized                     AS `Second Vendor Name Standardized`
        , second_bid_total_amount                             AS `Second Bid Total Amount`
        , second_project_bid_rank                             AS `Second Project Bid Rank`

        , benchmark_vendor_key                                AS `Benchmark Vendor Key`
        , benchmark_vendor_name                               AS `Benchmark Vendor Name`
        , benchmark_vendor_name_standardized                  AS `Benchmark Vendor Name Standardized`
        , benchmark_bid_total_amount                          AS `Benchmark Bid Total Amount`
        , benchmark_definition                                AS `Benchmark Definition`

        , bidder_count_in_project                             AS `Bidder Count In Project`
        , single_bidder_flag                                  AS `Single Bidder Flag`
        , competitive_project_flag                            AS `Competitive Project Flag`
        , lowest_bid_amount_in_project                        AS `Lowest Bid Amount In Project`
        , highest_bid_amount_in_project                       AS `Highest Bid Amount In Project`
        , bid_spread_amount_in_project                        AS `Bid Spread Amount In Project`

        , estimate_project_total                              AS `Estimate Project Total`
        , min_engineer_project_total                          AS `Min Engineer Project Total`
        , max_engineer_project_total                          AS `Max Engineer Project Total`
        , estimate_item_extended_total                        AS `Estimate Item Extended Total Diagnostic`
        , estimate_total_vs_item_rollup_abs_diff              AS `Header Vs Item Extended Total Abs Diff Diagnostic`

        , winner_vs_second_amount_diff                        AS `Winner Vs Second Amount Diff`
        , winner_vs_second_amount_ratio                       AS `Winner Vs Second Amount Ratio`
        , focus_vs_benchmark_amount_diff                      AS `Focus Vs Benchmark Amount Diff`
        , focus_vs_benchmark_amount_ratio                     AS `Focus Vs Benchmark Amount Ratio`
        , focus_vs_benchmark_outcome                          AS `Focus Vs Benchmark Outcome`
        , winner_vs_estimate_amount_diff                      AS `Winner Vs Estimate Amount Diff`
        , winner_vs_estimate_amount_ratio                     AS `Winner Vs Estimate Amount Ratio`
        , focus_vs_estimate_amount_diff                       AS `Focus Vs Estimate Amount Diff`
        , focus_vs_estimate_amount_ratio                      AS `Focus Vs Estimate Amount Ratio`

        , focus_under_estimate_flag                           AS `Focus Under Estimate Flag`
        , winner_under_estimate_flag                          AS `Winner Under Estimate Flag`
        , tight_bid_flag                                      AS `Tight Bid Flag`
        , missing_estimate_flag                               AS `Missing Estimate Flag`
        , has_second_place_bidder                             AS `Has Second Place Bidder`
        , winner_second_tie_flag                              AS `Winner Second Tie Flag`
        , focus_benchmark_tie_flag                            AS `Focus Benchmark Tie Flag`
        , focus_beats_second_flag                             AS `Focus Beats Second Flag`
        , winner_project_bid_rank_for_debug                   AS `Winner Project Bid Rank For Debug`
        , second_project_bid_rank_for_debug                   AS `Second Project Bid Rank For Debug`

        , item_comparison_row_count                           AS `Item Comparison Row Count`
        , focus_item_count                                    AS `Focus Item Count`
        , benchmark_item_count                                AS `Benchmark Item Count`
        , estimate_item_count                                 AS `Estimate Item Count`
        , matched_focus_benchmark_item_count                  AS `Matched Focus Benchmark Item Count`
        , matched_focus_estimate_item_count                   AS `Matched Focus Estimate Item Count`

        , focus_item_amount_total                             AS `Focus Item Amount Total`
        , benchmark_item_amount_total                         AS `Benchmark Item Amount Total`
        , estimate_item_amount_total                          AS `Estimate Item Amount Total`

        , focus_vs_benchmark_amount_sum                       AS `Focus Vs Benchmark Amount Sum`
        , focus_vs_benchmark_amount_avg                       AS `Focus Vs Benchmark Amount Avg`
        , focus_vs_benchmark_abs_amount_sum                   AS `Focus Vs Benchmark ABS Amount Sum`
        , focus_vs_benchmark_abs_amount_avg                   AS `Focus Vs Benchmark ABS Amount Avg`
        , focus_vs_benchmark_ratio_avg                        AS `Focus Vs Benchmark Ratio Avg`

        , overbid_item_count                                  AS `Overbid Item Count`
        , underbid_item_count                                 AS `Underbid Item Count`
        , tie_item_count                                      AS `Tie Item Count`

        , focus_vs_estimate_amount_sum                        AS `Focus Vs Estimate Amount Sum`
        , focus_vs_estimate_amount_avg                        AS `Focus Vs Estimate Amount Avg`

        , overbid_item_pct                                    AS `Overbid Item Pct`
        , underbid_item_pct                                   AS `Underbid Item Pct`
        , tie_item_pct                                        AS `Tie Item Pct`

        , focus_item_rollup_vs_focus_bid_total_ratio          AS `Focus Item Rollup Vs Focus Bid Total Ratio`
        , benchmark_item_rollup_vs_benchmark_bid_total_ratio  AS `Benchmark Item Rollup Vs Benchmark Bid Total Ratio`
        , estimate_item_rollup_vs_estimate_total_ratio        AS `Estimate Item Rollup Vs Estimate Total Ratio`

        , has_focus_vendor_county_history_flag                AS `Has Focus Vendor County History Flag`
        , has_focus_vendor_project_classification_history_flag AS `Has Focus Vendor Project Classification History Flag`
        , historical_low_focus_bid_for_project_classification AS `Historical Low Focus Bid For Project Classification`
        , historical_low_focus_bid_for_county_project_classification AS `Historical Low Focus Bid For County Project Classification`
        , is_project_opportunity_flag                         AS `Is Project Opportunity Flag`
        , focus_vendor_participated_opportunity_flag          AS `Focus Vendor Participated Opportunity Flag`
        , focus_vendor_missed_opportunity_flag                AS `Focus Vendor Missed Opportunity Flag`
    FROM {SOURCE_TABLE}
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_VIEW}").collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.vw_fact_project_bid_role_comparison successfully.")

    print("=" * 70)
    print("Built gold.vw_fact_project_bid_role_comparison")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise